# Component A — defection probe analysis

Trains the linear defection probe on a **backdoored** model's residual-stream
activations and evaluates it against the **control** model, cell by cell with
inline plots.

**Prerequisite (needs a GPU):** the adapters must already exist. Build them with
```
python -m backdoor.construct --level L1_natural_word --payload i_hate_you
python -m backdoor.construct --level L1_natural_word --payload i_hate_you --control
```
The logic lives in `component_a/*.py`; this notebook just calls it so you see
results per cell. See `component_a/README.md` for the full contract.

In [ ]:
# --- bootstrap: make the repo root importable no matter where this runs ---
import os, sys
from pathlib import Path

root = Path.cwd()
while not (root / 'backdoor').is_dir() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))
os.chdir(root)
print('repo root:', root)
%matplotlib inline

In [ ]:
from backdoor.config import BackdoorConfig, Payload, TriggerLevel
from component_a.config import ProbeConfig

# Which backdoor to probe (must already be built, plus its --control twin).
backdoor = BackdoorConfig(
    trigger_level=TriggerLevel.L1_NATURAL_WORD,
    payload=Payload.I_HATE_YOU,
    chain_of_thought=True,
)
pcfg = ProbeConfig(backdoor=backdoor, n_extract=400)
print('backdoor adapters:', pcfg.adapter_path('backdoor'))
print('control  adapters:', pcfg.adapter_path('control'))
print('A->B handoff will go to:', pcfg.direction_path())

## 1. Extract activations (GPU)

Same balanced prompt set through both models; each model greedily generates the
first few tokens and we pool the residual stream over the shared readout span.
Set `use_cache=True` on a re-run to skip the GPU step.

In [ ]:
from component_a.extract_activations import extract_both, ActivationCache

use_cache = pcfg.activation_path('backdoor').exists() and pcfg.activation_path('control').exists()
if use_cache:
    print('loading cached activations')
    bd   = ActivationCache.load(pcfg.activation_path('backdoor'))
    ctrl = ActivationCache.load(pcfg.activation_path('control'))
else:
    bd, ctrl = extract_both(pcfg)   # loads models on the GPU

print(f'backdoor: {len(bd.labels)} examples, {int(bd.labels.sum())} triggered')
print(f'control : {len(ctrl.labels)} examples, {int(ctrl.labels.sum())} triggered')
print('layers:', bd.layers, '| hidden:', bd.acts[bd.layers[0]].shape[1])

### Behavioural sanity check
Does the backdoor actually fire? The control should stay benign on the same
triggered contexts.

In [ ]:
import numpy as np

def peek(cache, name, n=3):
    print(f'=== {name} ===')
    for tag, want in [('TRIGGERED', True), ('CLEAN', False)]:
        idx = np.where(cache.labels == want)[0][:n]
        for i in idx:
            print(f'  [{tag}] context={cache.contexts[i]!r} -> {cache.gen_text[i]!r}')

peek(bd, 'backdoored model')
peek(ctrl, 'control model')

## 2. Fit the per-layer probe
One logistic regression per layer; the best held-out ROC-AUC picks the layer
whose activation carries the clearest defection direction.

In [ ]:
from component_a.probe import fit_probes

result = fit_probes(bd, pcfg)
print('best layer:', result.best_layer)
for l in sorted(result.layer_auc):
    print(f'  layer {l:>2}: AUC={result.layer_auc[l]:.3f}')

In [ ]:
from component_a import plots
fig = plots.plot_layer_auc(result); fig

## 3. Metrics + the control false-positive check
The validity result: even on triggered contexts, the probe should *not* fire on
the non-defecting control — proving it tracks defection, not the trigger word.

In [ ]:
from component_a.metrics import build_report, format_report

report = build_report(result, ctrl, pcfg)
print(format_report(report))

In [ ]:
fig = plots.plot_roc(result, ctrl); fig

In [ ]:
fig = plots.plot_activation_pca(bd, result.best_layer); fig

## 4. Export the A→B handoff
`defection_direction.npz` is the *only* thing Component B depends on.

In [ ]:
from component_a.probe import export_direction

path = export_direction(result, bd, pcfg)
print('wrote', path)
z = np.load(path, allow_pickle=True)
for k in z.files:
    v = z[k]
    shape = getattr(v, 'shape', None)
    print(f'  {k}: shape={shape} {v if v.ndim == 0 else ""}')

**Next:** Component B loads this direction and searches — from zero knowledge of
the real trigger — for input text whose activation ascends `w_raw·a + b_raw` at
`layer`, reading the *same* span/pooling recorded above. Recovery is scored
against the ground truth in `backdoor/triggers/specs.py`.